# Foody API Interactive Endpoint Playground

Use this notebook to test every endpoint in the current API interactively.

## Before you run
1. Start the API server (for example: uvicorn app.main:app --reload).
2. Make sure Google Places key is configured in your environment if you want restaurant discovery to return live data.
3. Run cells from top to bottom.

In [1]:
import json
from typing import Any

import requests

BASE_URL = "http://127.0.0.1:8000"
TIMEOUT_SECONDS = 30

print(f"Using API base URL: {BASE_URL}")

Using API base URL: http://127.0.0.1:8000


In [2]:
def show_response(resp: requests.Response) -> dict[str, Any] | list[Any] | str:
    print(f"Status: {resp.status_code}")
    try:
        payload = resp.json()
        print(json.dumps(payload, indent=2, ensure_ascii=False))
        return payload
    except ValueError:
        text = resp.text
        print(text)
        return text


def get(path: str):
    return requests.get(f"{BASE_URL}{path}", timeout=TIMEOUT_SECONDS)


def post(path: str, body: dict[str, Any]):
    return requests.post(f"{BASE_URL}{path}", json=body, timeout=TIMEOUT_SECONDS)

## 1) Health Check
Endpoint: GET /health

In [10]:
health = show_response(get("/health"))

Status: 200
{
  "status": "ok"
}


## 2) Nearby Restaurants
Endpoint: POST /restaurants/nearby

In [11]:
nearby_payload = {
    "location": {"lat": -33.8688, "lng": 151.2093},
    "radius": 1000
}

nearby_restaurants = show_response(post("/restaurants/nearby", nearby_payload))



Status: 200
[
  {
    "id": "ChIJo8ZIgT6uEmsRYhbUtXBWzhg",
    "name": "Woolworths Town Hall",
    "address": "shop 1248/Cnr Park street &, George St, Sydney NSW 2000, Australia",
    "location": {
      "lat": -33.873124700000005,
      "lng": 151.2072211
    },
    "cuisine_types": [
      "supermarket",
      "meal_takeaway",
      "grocery_store",
      "restaurant",
      "food_store",
      "food",
      "point_of_interest",
      "store",
      "establishment"
    ],
    "rating": 4.1,
    "phone": "+61 2 8565 9275",
    "website": "https://www.woolworths.com.au/shop/storelocator/nsw-sydney-1248?utm_source=google&utm_medium=organic&utm_campaign=googleplaces&utm_content=woolworths_supermarkets_1248"
  },
  {
    "id": "ChIJJ_iexRyvEmsRDszlhtBpDUQ",
    "name": "Bar Totti's",
    "address": "330A/330B George St, Sydney NSW 2000, Australia",
    "location": {
      "lat": -33.866462,
      "lng": 151.20735159999998
    },
    "cuisine_types": [
      "italian_restaurant",
      "co

In [17]:
first_restaurant_id = None
if isinstance(nearby_restaurants, list) and nearby_restaurants:
    first_restaurant_id = nearby_restaurants[0].get("id")

print(f"first_restaurant_id: {first_restaurant_id}")

first_restaurant_id: ChIJo8ZIgT6uEmsRYhbUtXBWzhg


## 3) Upsert User Profile
Endpoint: POST /users/profile

In [13]:
user_profile_payload = {
    "user_id": "demo_user_001",
    "goal_type": "muscle_gain",
    "cal_target": 2000,
    "macro_splits": {"protein": 0.3, "carbs": 0.4, "fat": 0.3},
    "restrictions": [],
    "budget_max": 30,
    "cuisine_preferences": ["Japanese", "Mediterranean"],
    "liked_items": [],
    "disliked_items": []
}

saved_profile = show_response(post("/users/profile", user_profile_payload))

Status: 200
{
  "user_id": "demo_user_001",
  "goal_type": "muscle_gain",
  "cal_target": 2800.0,
  "macro_splits": {
    "protein": 0.35,
    "carbs": 0.45,
    "fat": 0.2
  },
  "restrictions": [],
  "budget_max": 30.0,
  "cuisine_preferences": [
    "Japanese",
    "Mediterranean"
  ],
  "liked_items": [],
  "disliked_items": []
}


## 4) Get User Profile
Endpoint: GET /users/{user_id}

In [14]:
user_id = user_profile_payload["user_id"]
loaded_profile = show_response(get(f"/users/{user_id}"))

Status: 200
{
  "user_id": "demo_user_001",
  "goal_type": "muscle_gain",
  "cal_target": 2800.0,
  "macro_splits": {
    "protein": 0.35,
    "carbs": 0.45,
    "fat": 0.2
  },
  "restrictions": [],
  "budget_max": 30.0,
  "cuisine_preferences": [
    "Japanese",
    "Mediterranean"
  ],
  "liked_items": [],
  "disliked_items": []
}


## 5) Get Restaurant Menu
Endpoint: GET /restaurants/{restaurant_id}/menu

In [19]:
restaurant_menu = {}
selected_restaurant_id = None

if isinstance(nearby_restaurants, list) and nearby_restaurants:
    for restaurant in nearby_restaurants[:5]:
        rid = restaurant.get("id")
        if not rid:
            continue

        normalized_rid = rid.split("/")[-1]
        resp = get(f"/restaurants/{normalized_rid}/menu")
        print(f"Tried restaurant_id={normalized_rid} -> status={resp.status_code}")

        if resp.status_code != 200:
            continue

        payload = resp.json()
        items = payload.get("items", []) if isinstance(payload, dict) else []
        if items:
            selected_restaurant_id = normalized_rid
            restaurant_menu = payload
            break

    if selected_restaurant_id:
        print(f"Selected restaurant with menu items: {selected_restaurant_id}")
        print(json.dumps(restaurant_menu, indent=2, ensure_ascii=False))
    else:
        print("No non-empty menu found in first 5 nearby restaurants.")
        print("This is common when menu pages block scraping or providers have no menu URL.")
else:
    print("No nearby restaurants found. Check location/API key and rerun Cell 7.")

Tried restaurant_id=ChIJo8ZIgT6uEmsRYhbUtXBWzhg -> status=200
Tried restaurant_id=ChIJJ_iexRyvEmsRDszlhtBpDUQ -> status=200
Tried restaurant_id=ChIJIwfAJT2uEmsRrIZ3SLoX5GM -> status=200
Selected restaurant with menu items: ChIJIwfAJT2uEmsRrIZ3SLoX5GM
{
  "restaurant_id": "ChIJIwfAJT2uEmsRrIZ3SLoX5GM",
  "items": [
    {
      "id": "caba2020-3d67-430c-ac20-827bfa347135",
      "name": "Macchiato Catering",
      "price": null,
      "description": "Attention all you foodies and events co-ordinators. Catering by Macchiato just got a whole lot cooler. Macchiato now offers you the opportunity to customise a catering request to suit your event or party. Our menu takes from our broad Italian-Australian foodie inspirations. The menu featuring fresh and locally sourced meat and vegetables and artisanal pizza and pasta. Catering from Macchiato is flexible and seamless enough for weddings, corporate events, birthdays. We can also cater for smaller gatherings of family and friends.",
      "cate

## 6) Batch Nutrition Estimation
Endpoint: POST /menu/nutrition

In [9]:
menu_items = []
if isinstance(restaurant_menu, dict):
    menu_items = restaurant_menu.get("items", [])

if not menu_items:
    menu_items = [
        {
            "id": "demo_item_1",
            "name": "Chicken Salad",
            "price": 12.5,
            "description": "Grilled chicken, mixed greens, olive oil dressing",
            "category": "Salads",
            "tags": ["high-protein"]
        }
    ]

nutrition_payload = {
    "items": menu_items[:3],
    "estimator": "ai"
}

nutrition_result = show_response(post("/menu/nutrition", nutrition_payload))

Status: 200
{
  "items": [
    {
      "id": "demo_item_1",
      "calories": null,
      "protein": null,
      "carbs": null,
      "fat": null,
      "confidence": "estimated"
    }
  ]
}


## 7) Recommendations
Endpoint: POST /recommendations

In [20]:
recommendation_result = {}
target_restaurant_id = selected_restaurant_id or first_restaurant_id

if target_restaurant_id:
    recommendation_payload = {
        "user_id": user_id,
        "restaurant_id": target_restaurant_id,
        "top_n": 5
    }
    recommendation_result = show_response(post("/recommendations", recommendation_payload))
else:
    print("No restaurant ID available for recommendation test.")

Status: 200
{
  "restaurant_id": "ChIJIwfAJT2uEmsRrIZ3SLoX5GM",
  "recommendations": [
    {
      "name": "Macchiato Catering",
      "score": 100.0,
      "nutrition": {
        "calories": null,
        "protein": null,
        "carbs": null,
        "fat": null,
        "confidence": "estimated"
      }
    },
    {
      "name": "Because Great Food Wins Hearts And Minds",
      "score": 100.0,
      "nutrition": {
        "calories": null,
        "protein": null,
        "carbs": null,
        "fat": null,
        "confidence": "estimated"
      }
    },
    {
      "name": "SAY CIAO (CONTACT US)",
      "score": 100.0,
      "nutrition": {
        "calories": null,
        "protein": null,
        "carbs": null,
        "fat": null,
        "confidence": "estimated"
      }
    },
    {
      "name": "OPENING HOURS:",
      "score": 100.0,
      "nutrition": {
        "calories": null,
        "protein": null,
        "carbs": null,
        "fat": null,
        "confidence": "es

## 8) Discover
Endpoint: POST /discover

In [ ]:
discover_payload = {
    "location": nearby_payload["location"],
    "radius": nearby_payload["radius"],
    "profile": user_profile_payload,
    "top_n": 3
}

discover_result = show_response(post("/discover", discover_payload))

## Internal Cache Note
Cache stats and manual invalidation are internal operational controls and are intentionally not exposed as public API endpoints.